In [66]:
 !pip install scikit-fingerprints  # Package 'skfp' could not be found

In [67]:
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
import torch
from skfp.fingerprints import ECFPFingerprint # Temporarily removed due to installation error

In [68]:

# ==========================================
# FUNKCJA: Parsowanie hierarchii z pliku .obo
# ==========================================
def parse_obo_hierarchy(file_path):
    """
    Czyta plik .obo i zwraca słownik relacji: { 'Rodzic': ['Dziecko1', 'Dziecko2', ...] }
    """
    parent_child_map = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        current_term = None
        for line in f:
            line = line.strip()
            if line.startswith('[Term]'):
                current_term = None
            elif line.startswith('id:'):
                current_term = line.split('id:')[1].strip()
                if current_term not in parent_child_map:
                    parent_child_map[current_term] = []
            elif line.startswith('is_a:'):
                # Format wpisu to często: "is_a: CHEBI:XXXX ! nazwa"
                parent = line.split('is_a:')[1].split('!')[0].strip()
                if parent not in parent_child_map:
                    parent_child_map[parent] = []
                if current_term:
                    parent_child_map[parent].append(current_term)
    return parent_child_map

In [69]:
# ==========================================
# KROK 1: Wczytanie danych
# ==========================================
print("1. Wczytywanie danych...")
train_df = pd.read_parquet('Hackathon_Task_1/Hackathon_task_1/data/chebi_dataset_train.parquet')
test_df = pd.read_parquet('Hackathon_Task_1/Hackathon_task_1/data/chebi_dataset_test_empty.parquet')

# Wyodrębnienie kolumn: SMILES i etykiet klas (zakładamy, że reszta to klasy CHEBI)
smiles_train = train_df['SMILES'].tolist()
y_train = train_df.drop(columns=['SMILES', 'mol_id']).values
class_names = train_df.drop(columns=['SMILES', 'mol_id']).columns.tolist()

smiles_test = test_df['SMILES'].tolist()

1. Wczytywanie danych...


In [70]:
smiles_train[:2]

['CCCCC/C=C\\CCCCCCCC(=O)O', 'Cc1cc2cc(O)cc(O)c2c(C)n1']

In [71]:
y_train.shape

(33668, 500)

In [72]:
y_train[y_train==1].shape

(926722,)

In [73]:
y_train[y_train==0].shape

(15907278,)

In [74]:
def get_unique_characters(smiles_list):
    """
    Scans the entire dataset and returns a sorted list of all unique characters.
    """
    unique_chars = set()
    for smi in smiles_list:
        # list(smi) breaks the string into individual characters
        unique_chars.update(list(smi))

    return sorted(list(unique_chars))

# Assuming 'smiles_train' is your full list of training SMILES
all_chars = get_unique_characters(smiles_train)
print(f"Total unique characters found: {len(all_chars)}")
print("Characters:", "".join(all_chars))

Total unique characters found: 67
Characters: #%()+-./0123456789=@ABCDEFGHIKLMNOPRSTUVWXYZ[\]abcdefghilmnoprstuvy


In [75]:
from tokenizers import Tokenizer, models, trainers
from tokenizers.processors import TemplateProcessing

def train_bpe_tokenizer(smiles_list, vocab_size=1000):
    """
    Trains a Byte-Pair Encoding tokenizer specifically on your SMILES data.
    """
    print("Initializing BPE Tokenizer...")

    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]"],
        show_progress=True
    )

    print(f"Training BPE to a vocabulary size of {vocab_size}...")
    tokenizer.train_from_iterator(smiles_list, trainer=trainer)

    tokenizer.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]",
        pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[
            ("[CLS]", tokenizer.token_to_id("[CLS]")),
            ("[SEP]", tokenizer.token_to_id("[SEP]")),
        ],
    )

    # FIXED: Added explicit padding configuration to ensure all tensors have the same length
    tokenizer.enable_padding(pad_id=tokenizer.token_to_id("[PAD]"), pad_token="[PAD]", length=128)
    tokenizer.enable_truncation(max_length=128)

    return tokenizer

bpe_tokenizer = train_bpe_tokenizer(smiles_train, vocab_size=800)
bpe_tokenizer.save("chebi_smiles_bpe.json")
print("BPE Tokenizer trained and saved successfully with fixed padding!")

Initializing BPE Tokenizer...
Training BPE to a vocabulary size of 800...
BPE Tokenizer trained and saved successfully with fixed padding!


In [76]:
# Test it on the first molecule in your dataset
sample_smi = smiles_train[0]
encoded = bpe_tokenizer.encode(sample_smi)

print(f"Original SMILES: {sample_smi}")
print(f"Tokens: {encoded.tokens}")
print(f"Token IDs: {encoded.ids}")

Original SMILES: CCCCC/C=C\CCCCCCCC(=O)O
Tokens: ['[CLS]', 'CCCC', 'C/C=C\\', 'CCCCCCCC(=O)', 'O', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '

In [77]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
from tokenizers import Tokenizer

# ==========================================
# 1. Train/Validation Split (90:10)
# ==========================================
smiles_train_split, smiles_val_split, y_train_split, y_val_split = train_test_split(
    smiles_train, y_train, test_size=0.10, random_state=42
)
print(f"Train size: {len(smiles_train_split)}, Val size: {len(smiles_val_split)}")

# ==========================================
# 2. Load the BPE Tokenizer
# ==========================================
# Load the tokenizer we trained and saved in the previous step
tokenizer = Tokenizer.from_file("chebi_smiles_bpe.json")
print(f"BPE Vocabulary size: {tokenizer.get_vocab_size()}")

# ==========================================
# 3. Custom PyTorch Dataset (Updated for BPE)
# ==========================================
class ChEBIDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer):
        self.smiles = smiles_list
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smi = self.smiles[idx]

        # The HF tokenizer automatically applies max_len, padding, and special tokens
        encoded = self.tokenizer.encode(smi)

        token_ids = torch.tensor(encoded.ids, dtype=torch.long)

        # attention_mask is 1 for real tokens and 0 for padding tokens
        mask = torch.tensor(encoded.attention_mask, dtype=torch.bool)

        # Convert labels to float32 for BCE/Focal Loss
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.float32)

        return token_ids, mask, label_tensor

# ==========================================
# 4. Create DataLoaders
# ==========================================
BATCH_SIZE = 64
# Note: max_len is now handled strictly inside the saved tokenizer config

train_dataset = ChEBIDataset(smiles_train_split, y_train_split, tokenizer)
val_dataset = ChEBIDataset(smiles_val_split, y_val_split, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 5. Small Transformer Model
# ==========================================
class SmallSMILESTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=3, num_classes=500):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        # Simple learnable positional encoding
        self.pos_encoder = nn.Parameter(torch.zeros(1, 512, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head for 500 binary outputs
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x, mask):
        seq_len = x.size(1)

        out = self.embedding(x) + self.pos_encoder[:, :seq_len, :]

        # PyTorch transformer takes padding mask where True means "ignore"
        # Our mask has 1 for real tokens, 0 for pad. So we invert it.
        padding_mask = ~mask

        out = self.transformer(out, src_key_padding_mask=padding_mask)

        # Global Average Pooling (ignoring padding tokens)
        mask_float = mask.unsqueeze(-1).float()
        out = (out * mask_float).sum(dim=1) / mask_float.sum(dim=1)

        return self.fc(out)

# ==========================================
# 6. Binary Focal Loss for Multi-label
# ==========================================
class BinaryFocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)

        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ==========================================
# 7. Initialization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dynamically get the vocabulary size from the BPE tokenizer
vocab_size = tokenizer.get_vocab_size()

model = SmallSMILESTransformer(vocab_size=vocab_size).to(device)
criterion = BinaryFocalLossWithLogits(alpha=0.78, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("Model architecture ready and moved to:", device)

Train size: 30301, Val size: 3367
BPE Vocabulary size: 800
Model architecture ready and moved to: cuda


In [78]:
import torch
torch.cuda.is_available()

True

In [79]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
from tokenizers import Tokenizer

# ==========================================
# 1. Train/Validation Split (90:10)
# ==========================================
smiles_train_split, smiles_val_split, y_train_split, y_val_split = train_test_split(
    smiles_train, y_train, test_size=0.10, random_state=42
)
print(f"Train size: {len(smiles_train_split)}, Val size: {len(smiles_val_split)}")

# ==========================================
# 2. Load the BPE Tokenizer
# ==========================================
# Load the tokenizer we trained and saved in the previous step
tokenizer = Tokenizer.from_file("chebi_smiles_bpe.json")
print(f"BPE Vocabulary size: {tokenizer.get_vocab_size()}")

# ==========================================
# 3. Custom PyTorch Dataset (Updated for BPE)
# ==========================================
class ChEBIDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer):
        self.smiles = smiles_list
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smi = self.smiles[idx]

        # The HF tokenizer automatically applies max_len, padding, and special tokens
        encoded = self.tokenizer.encode(smi)

        token_ids = torch.tensor(encoded.ids, dtype=torch.long)

        # attention_mask is 1 for real tokens and 0 for padding tokens
        mask = torch.tensor(encoded.attention_mask, dtype=torch.bool)

        # Convert labels to float32 for BCE/Focal Loss
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.float32)

        return token_ids, mask, label_tensor

# ==========================================
# 4. Create DataLoaders
# ==========================================
BATCH_SIZE = 64
# Note: max_len is now handled strictly inside the saved tokenizer config

train_dataset = ChEBIDataset(smiles_train_split, y_train_split, tokenizer)
val_dataset = ChEBIDataset(smiles_val_split, y_val_split, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 5. Small Transformer Model
# ==========================================
class SmallSMILESTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=3, num_classes=500):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        # Simple learnable positional encoding
        self.pos_encoder = nn.Parameter(torch.zeros(1, 512, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head for 500 binary outputs
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x, mask):
        seq_len = x.size(1)

        out = self.embedding(x) + self.pos_encoder[:, :seq_len, :]

        # PyTorch transformer takes padding mask where True means "ignore"
        # Our mask has 1 for real tokens, 0 for pad. So we invert it.
        padding_mask = ~mask

        out = self.transformer(out, src_key_padding_mask=padding_mask)

        # Global Average Pooling (ignoring padding tokens)
        mask_float = mask.unsqueeze(-1).float()
        out = (out * mask_float).sum(dim=1) / mask_float.sum(dim=1)

        return self.fc(out)

# ==========================================
# 6. Binary Focal Loss for Multi-label
# ==========================================
class BinaryFocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)

        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ==========================================
# 7. Initialization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dynamically get the vocabulary size from the BPE tokenizer
vocab_size = tokenizer.get_vocab_size()

model = SmallSMILESTransformer(vocab_size=vocab_size).to(device)
criterion = BinaryFocalLossWithLogits(alpha=0.7, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("Model architecture ready and moved to:", device)

Train size: 30301, Val size: 3367
BPE Vocabulary size: 800
Model architecture ready and moved to: cuda


In [80]:
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm
import numpy as np
import torch

EPOCHS = 50
PATIENCE = 4

best_val_f1 = 0
epochs_without_improvement = 0

print("Starting Training...")

for epoch in range(EPOCHS):

    # ==========================
    # TRAINING
    # ==========================
    model.train()
    total_train_loss = 0.0
    train_preds = []
    train_targets = []

    train_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for batch_tokens, batch_mask, batch_labels in train_iterator:
        batch_tokens = batch_tokens.to(device)
        batch_mask = batch_mask.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        logits = model(batch_tokens, batch_mask)
        loss = criterion(logits, batch_labels)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int().cpu().numpy()

        train_preds.append(preds)
        train_targets.append(batch_labels.cpu().numpy())

        train_iterator.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)
    train_f1 = f1_score(np.vstack(train_targets), np.vstack(train_preds), average="macro")

    # ==========================
    # VALIDATION
    # ==========================
    model.eval()

    total_val_loss = 0.0
    val_preds = []
    val_targets = []

    with torch.no_grad():
        for batch_tokens, batch_mask, batch_labels in val_loader:

            batch_tokens = batch_tokens.to(device)
            batch_mask = batch_mask.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_tokens, batch_mask)

            v_loss = criterion(logits, batch_labels)
            total_val_loss += v_loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int().cpu().numpy()

            val_preds.append(preds)
            val_targets.append(batch_labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_f1 = f1_score(np.vstack(val_targets), np.vstack(val_preds), average="macro")

    # ==========================
    # SUMMARY
    # ==========================
    print(f"\n--- Epoch {epoch+1} Summary ---")
    print(f"Train - Loss: {avg_train_loss:.4f}, Macro F1: {train_f1:.4f}")
    print(f"Valid - Loss: {avg_val_loss:.4f}, Macro F1: {val_f1:.4f}")

    # ==========================
    # EARLY STOPPING
    # ==========================
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        epochs_without_improvement = 0

        torch.save(model.state_dict(), "best_model.pt")

        print("New best model saved.")

    else:

        epochs_without_improvement += 1
        print(f"No improvement for {epochs_without_improvement} epoch(s)")

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break


# ==========================
# LOAD BEST MODEL
# ==========================

model.load_state_dict(torch.load("best_model.pt"))

print("\nBest model loaded. Training complete.")


Starting Training...


Epoch 1/50 [Train]:   0%|          | 0/474 [00:00<?, ?it/s]


--- Epoch 1 Summary ---
Train - Loss: 0.0355, Macro F1: 0.0405
Valid - Loss: 0.0250, Macro F1: 0.0258
New best model saved.


Epoch 2/50 [Train]:   0%|          | 0/474 [00:00<?, ?it/s]


--- Epoch 2 Summary ---
Train - Loss: 0.0239, Macro F1: 0.0403
Valid - Loss: 0.0204, Macro F1: 0.0719
New best model saved.


...


--- Epoch 48 Summary ---
Train - Loss: 0.0045, Macro F1: 0.7333
Valid - Loss: 0.0072, Macro F1: 0.7015
No improvement for 2 epoch(s)


Epoch 49/50 [Train]:   0%|          | 0/474 [00:00<?, ?it/s]


--- Epoch 49 Summary ---
Train - Loss: 0.0044, Macro F1: 0.7362
Valid - Loss: 0.0073, Macro F1: 0.7081
New best model saved.


Epoch 50/50 [Train]:   0%|          | 0/474 [00:00<?, ?it/s]


--- Epoch 50 Summary ---
Train - Loss: 0.0044, Macro F1: 0.7396
Valid - Loss: 0.0073, Macro F1: 0.7023
No improvement for 1 epoch(s)

Best model loaded. Training complete.


# First model

In [103]:
import numpy as np
from sklearn.decomposition import PCA
from skfp.fingerprints import ECFPFingerprint

print("1. Generating raw 2048-dim ECFP for Train and Validation sets...")
# Initialize the generator
ecfp_generator = ECFPFingerprint(radius=2, fp_size=2048, n_jobs=-1)

# Transform the SMILES lists into raw matrices
raw_train_ecfp = ecfp_generator.transform(smiles_train_split)
raw_val_ecfp = ecfp_generator.transform(smiles_val_split)

print("2. Fitting PCA on Training Data to compress to 256 dimensions...")
# Initialize PCA to keep 256 components
pca = PCA(n_components=256, random_state=42)

# FIT ONLY ON TRAIN, then transform both
train_ecfp_pca = pca.fit_transform(raw_train_ecfp)
val_ecfp_pca = pca.transform(raw_val_ecfp)

# Optional: Print how much variance we kept!
explained_variance = np.sum(pca.explained_variance_ratio_)
print(f"PCA completed! We retained {explained_variance * 100:.2f}% of the original structural variance.")

1. Generating raw 2048-dim ECFP for Train and Validation sets...
2. Fitting PCA on Training Data to compress to 256 dimensions...
PCA completed! We retained 72.29% of the original structural variance.


In [104]:
from rdkit import Chem

# 1. Define your SMARTS pattern
smarts_pattern = Chem.MolFromSmarts('[CX3](=O)[OX2H1]')

# 2. Create a test molecule that SHOULD match (Acetic Acid)
test_mol_positive = Chem.MolFromSmiles('CC(=O)O')

# 3. Create a test molecule that SHOULD NOT match (Ethanol)
test_mol_negative = Chem.MolFromSmiles('CCO')

# 4. Test them!
print("Matches Acetic Acid?", test_mol_positive.HasSubstructMatch(smarts_pattern)) # Should print True
print("Matches Ethanol?", test_mol_negative.HasSubstructMatch(smarts_pattern))     # Should print False

Matches Acetic Acid? True
Matches Ethanol? False


In [105]:
from skfp.fingerprints import MACCSFingerprint

print("Bulk computing MACCS keys...")
# Initialize with n_jobs=-1 to use all cores
maccs_gen = MACCSFingerprint(n_jobs=-1)

# Transform entire lists at once
train_maccs = maccs_gen.transform(smiles_train_split)
val_maccs = maccs_gen.transform(smiles_val_split)

print(f"MACCS extraction complete! Shape: {train_maccs.shape}")

Bulk computing MACCS keys...
MACCS extraction complete! Shape: (30301, 166)


In [112]:
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors

def get_advanced_descriptors(mol):
    if mol is None:
        return np.zeros(10)

    # 7 PhysChem Descriptors
    wt = Descriptors.MolWt(mol) / 1000            # Normalized Weight
    logp = Descriptors.MolLogP(mol) / 10          # Hydrophobicity
    hbd = Lipinski.NumHDonors(mol) / 10           # H-Bond Donors
    hba = Lipinski.NumHAcceptors(mol) / 10        # H-Bond Acceptors
    tpsa = Descriptors.TPSA(mol) / 200            # Polar Surface Area
    fsp3 = Lipinski.FractionCSP3(mol)             # Complexity (sp3 fraction)

    # 4 Topological/Ring Descriptors (Crucial for ChEBI)
    rings = Lipinski.RingCount(mol) / 10          # Total Rings
    arom_rings = Lipinski.NumAromaticRings(mol) / 10
    hetero_atoms = Lipinski.NumHeteroatoms(mol) / 20
    # Bridgehead atoms define complex polycyclic systems
    bridgeheads = rdMolDescriptors.CalcNumBridgeheadAtoms(mol) / 5

    return np.array([wt, logp, hbd, hba, tpsa, fsp3, rings, arom_rings, hetero_atoms, bridgeheads])

In [113]:
class OptimizedOmniDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, precomputed_pca, precomputed_maccs):
        self.smiles = smiles_list
        self.labels = labels
        self.tokenizer = tokenizer
        self.ecfp_pca = precomputed_pca
        self.maccs = precomputed_maccs
        self.smarts_carboxylic = Chem.MolFromSmarts('[CX3](=O)[OX2H1]')

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smi = self.smiles[idx]

        # --- A. Text Tokenization ---
        encoded = self.tokenizer.encode(smi)
        token_ids = torch.tensor(encoded.ids, dtype=torch.long)
        mask = torch.tensor(encoded.attention_mask, dtype=torch.bool)

        # --- B. Precomputed Lookups ---
        ecfp_feat = self.ecfp_pca[idx]
        maccs_feat = self.maccs[idx]

        # --- C. On-the-fly Structural Features ---
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                # Custom SMARTS
                has_carb = 1.0 if mol.HasSubstructMatch(self.smarts_carboxylic) else 0.0
                # New Advanced Descriptors
                adv_desc = get_advanced_descriptors(mol)
                structural_feats = np.concatenate([[has_carb], adv_desc])
            else:
                structural_feats = np.zeros(11) # 1 SMARTS + 10 Descriptors
        except Exception:
            structural_feats = np.zeros(11)

        # --- D. Final Concatenation (433 total dims) ---
        combined_features = np.concatenate([
            ecfp_feat,
            maccs_feat,
            structural_feats
        ])

        return (
            token_ids,
            mask,
            torch.tensor(combined_features, dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

In [114]:
BATCH_SIZE = 64

# Pass the precomputed PCA and MACCS arrays directly into the dataset
train_dataset = OptimizedOmniDataset(smiles_train_split, y_train_split, tokenizer, train_ecfp_pca, train_maccs)
val_dataset = OptimizedOmniDataset(smiles_val_split, y_val_split, tokenizer, val_ecfp_pca, val_maccs)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Check the new dynamic feature dimension to pass to your model
sample_tokens, sample_mask, sample_feats, sample_labels = train_dataset[0]
DENSE_FEATURE_DIM = sample_feats.shape[0]

print(f"Total Fast Feature Dimension per molecule: {DENSE_FEATURE_DIM}")

Total Fast Feature Dimension per molecule: 433


In [115]:
import torch.nn as nn

class OmniFusionTransformer(nn.Module):
    def __init__(self, vocab_size, dense_feature_dim, d_model=128, nhead=4, num_layers=3, num_classes=500, max_seq_len=512):
        super().__init__()

        # --- 1. Transformer Branch (Sequence Learning) ---
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model))

        # --- 2. Omni-Feature Branch (PCA + MACCS + Descriptors + SMARTS) ---
        self.feature_norm = nn.BatchNorm1d(dense_feature_dim)

        self.dense_fc = nn.Sequential(
            nn.Linear(dense_feature_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.GELU()
        )

        # --- 3. Fusion & Classification Head ---
        fusion_dim = d_model + 256
        self.fc_fusion = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim * 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(fusion_dim * 2, num_classes)
        )

    def forward(self, x, mask, dense_features):
        # A. Process Sequence
        seq_len = x.size(1)
        out_trans = self.embedding(x) + self.pos_encoder[:, :seq_len, :]

        padding_mask = (mask == 0)
        out_trans = self.transformer(out_trans, src_key_padding_mask=padding_mask)

        mask_float = mask.unsqueeze(-1).float()
        out_trans = (out_trans * mask_float).sum(dim=1) / mask_float.sum(dim=1).clamp(min=1e-9)

        # B. Process Omni-Features
        out_dense = self.feature_norm(dense_features)
        out_dense = self.dense_fc(out_dense)

        # C. Concatenate and Classify
        combined = torch.cat((out_trans, out_dense), dim=1)

        return self.fc_fusion(combined)

# Initialize the new model
model = OmniFusionTransformer(
    vocab_size=tokenizer.get_vocab_size(),
    dense_feature_dim=DENSE_FEATURE_DIM # This dynamically adapts to your PCA dimension!
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print("Omni-Fusion Model ready and moved to:", device)

Omni-Fusion Model ready and moved to: cuda


In [ ]:
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm
import numpy as np
import torch
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.warning')

# --- CONFIGURATION ---
EPOCHS = 100
PRIMARY_PATIENCE = 3      # How many epochs to wait before reducing LR
MAX_LR_REDUCTIONS = 1     # How many times we can reduce LR (e.g., 1 time means LR/10 then stop)
best_val_f1 = 0
epochs_without_improvement = 0
lr_reductions_count = 0

print(f"Starting Training (LR: {optimizer.param_groups[0]['lr']})...")

for epoch in range(EPOCHS):
    # ==========================
    # TRAINING
    # ==========================
    model.train()
    total_train_loss = 0.0
    train_preds, train_targets = [], []

    train_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for batch_tokens, batch_mask, batch_features, batch_labels in train_iterator:
        batch_tokens, batch_mask = batch_tokens.to(device), batch_mask.to(device)
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        optimizer.zero_grad()
        logits = model(batch_tokens, batch_mask, batch_features)

        # NOTE: Use a weighted loss here if possible to handle imbalanced child classes
        loss = criterion(logits, batch_labels)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).int().cpu().numpy()

        train_preds.append(preds)
        train_targets.append(batch_labels.cpu().numpy())
        train_iterator.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)
    train_f1 = f1_score(np.vstack(train_targets), np.vstack(train_preds), average="macro")

    # ==========================
    # VALIDATION
    # ==========================
    model.eval()
    total_val_loss = 0.0
    val_preds, val_targets = [], []

    with torch.no_grad():
        for batch_tokens, batch_mask, batch_features, batch_labels in val_loader:
            batch_tokens, batch_mask = batch_tokens.to(device), batch_mask.to(device)
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            logits = model(batch_tokens, batch_mask, batch_features)
            v_loss = criterion(logits, batch_labels)

            total_val_loss += v_loss.item()
            preds = (torch.sigmoid(logits) > 0.5).int().cpu().numpy()

            val_preds.append(preds)
            val_targets.append(batch_labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_f1 = f1_score(np.vstack(val_targets), np.vstack(val_preds), average="macro")

    # ==========================
    # LOGGING & SCHEDULING
    # ==========================
    print(f"\n--- Epoch {epoch+1} Summary ---")
    print(f"Train - Loss: {avg_train_loss:.4f}, Macro F1: {train_f1:.4f}")
    print(f"Valid - Loss: {avg_val_loss:.4f}, Macro F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "best_model_omni.pt")
        print(f"🔥 New best model saved (F1: {val_f1:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"⚠️ No improvement for {epochs_without_improvement} epoch(s)")

        # --- LR REDUCTION LOGIC ---
        if epochs_without_improvement >= PRIMARY_PATIENCE:
            if lr_reductions_count < MAX_LR_REDUCTIONS:
                lr_reductions_count += 1
                epochs_without_improvement = 0 # Reset patience for the new LR

                # Reduce LR by factor of 10
                for param_group in optimizer.param_groups:
                    param_group['lr'] /= 10.0

                new_lr = optimizer.param_groups[0]['lr']
                print(f"📉 Reducing LR to {new_lr}. Resetting patience.")

                # Optional: Reload best weights before starting with lower LR
                model.load_state_dict(torch.load("best_model_omni.pt"))
            else:
                print("🛑 Final patience reached after LR reduction. Ending training.")
                break

# Load best weights for final use
model.load_state_dict(torch.load("best_model_omni.pt"))
print("\nBest model reloaded. Ready for evaluation.")

Starting Training (LR: 0.0001)...


Epoch 1/100 [Train]:   0%|          | 0/474 [00:00<?, ?it/s]